# Expand master parameter list

This is a **utility** notebook — rerun it whenever the parameter sweep changes, not
something you'd read start to end (see the section overview). Its one job: expand a
base parameter spec into the full table of `(E, τ, lag, knn, surrogate)` combinations
that `CCMConfig` reads to run one CCM calculation per row. That's the machinery
behind the 49 E–τ configurations and ±40-decade lag scan the paper reports (Results;
Methods: *Workflow*) — see [A Single CCM Calculation](../2_CCM_single_dyad/1_run__CCMlocally.ipynb) for what a single row of this
table actually does. Each cell below expands one slice of the grid for one dyad; get
the full 49-configuration sweep (E = 4–10, τ = 1–8) by rerunning this notebook (or
the batch script it mirrors) across all four dyads.


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from operator import index
from pathlib import Path
import os
import sys
import itertools


In [5]:
stem = Path(*(p := Path.cwd().resolve()).parts[: p.parts.index("notebooks")])
proj_stem = stem / 'hol_temp_tsi_ccm'
print(proj_stem)

/Users/jlanders/PycharmProjects/hol_temp_tsi_ccm_pb/hol_temp_tsi_ccm


In [6]:
import numpy as np
import pandas as pd
pd.option_context('mode.use_inf_as_na', True)

import warnings
warnings.filterwarnings("ignore", category=UserWarning)#, module='seaborn')
warnings.filterwarnings("ignore", category=FutureWarning)#, module='seaborn')
warnings.simplefilter("ignore", category=FutureWarning)

from cedarkit.core.project_config import load_config
from cedarkit.utils.routing.paths import *
from cedarkit.utils.routing.file_name_parsers import * #data_access import *
from cedarkit.utils.workflow.parameter_utils import *


In [7]:
dyad_name = 'GISP2Alley00Tanom_Wu18TSI'
dyad_name = 'Erb22daGMST_Wu18TSI'

In [8]:
dyad_dir = proj_stem/f'{dyad_name}'#Path(os.getcwd()).resolve().parents[0]
config = load_config(dyad_dir / 'proj_config.yaml')

calc_location =set_calc_path(None, dyad_dir, config, '')
calc_location.mkdir(parents=True, exist_ok=True)

output_location = set_output_path(None, calc_location, config)
output_location.mkdir(parents=True, exist_ok=True)



In [10]:
import importlib
parameters_dir = dyad_dir / 'parameters'
spec = importlib.util.spec_from_file_location("parameters", parameters_dir / f'{config.parameters.spec_d}.py')
params_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(params_module)

# Now you can access the parameters dictionary
parameters_d_master = params_module.parameters_d

In [14]:
def general_lag_combinations(col_var_id, target_var_id, E_vals, tau_vals, lag_vals, config, parameters_d_master, surr_nums=[0,1], surr_vars=['neither']):
    param_csv_path = dyad_dir / 'parameters' / f'params_bycol_{col_var_id}.csv'

    parameters_d = parameters_d_master.copy()
    parameters_d['E']['values'] = E_vals
    parameters_d['tau']['values'] = tau_vals
    parameters_d['lag']['values'] = lag_vals
    col_var_alias = config.col.var
    target_var_alias = config.target.var

    var_id_tuple= (col_var_id, col_var_alias, target_var_id, target_var_alias,surr_vars, surr_nums,[], param_csv_path, parameters_d)
    return var_id_tuple, param_csv_path


# Sweep-style combinations

`E_vals` and `tau_vals` here cover one dyad's slice of the parameter sweep (across
all four dyads, E ranges 4–10 and τ ranges 1–8 — the 49 configurations from
Results). Setting `surr_vars=['neither']` with a single `surr_num` tells this batch
to compute the **real** relationship only; generate the matching surrogate
relationships (Text S1.3: 200 replicates per direction) by rerunning with
`surr_vars` set to the column/target variable names instead.


In [15]:
col_var_ids = [config.col.var_id]
target_var_id = config.target.var_id

E_vals = np.arange(5, 8, 1) #np.arange(3, 10, 2) #np.arange(3, 9, 2)
tau_vals = np.arange(1, 6, 1) #np.arange(3, 10, 2) #np.arange(3, 9, 2)


In [16]:
surr_nums, surr_vars, lag_vals = [0], ['neither'], np.arange(-41, 41, 1)
# surr_nums, surr_vars, lag_vals = [1, 201], [config.col.var, config.target.var], [0]
lag_comb_info = [general_lag_combinations(
    col_var_id=col_var_id,
    target_var_id=target_var_id,
    E_vals=E_vals,
    tau_vals=tau_vals,
    lag_vals=lag_vals,
    config=config,
    parameters_d_master=parameters_d_master,
    surr_nums=surr_nums,#[1, 201],
    surr_vars=surr_vars#config.surr_vars
) for col_var_id in col_var_ids]


In [17]:
param_csv_paths = []
for var_id_tuple, param_csv_path in lag_comb_info:
    process_params_group(var_id_tuple)
    param_csv_paths.append(param_csv_path)


In [18]:
tidy_up_params(param_csv_paths, keep='first')


/Users/jlanders/PycharmProjects/hol_temp_tsi_ccm_pb/hol_temp_tsi_ccm/Erb22daGMST_Wu18TSI/parameters/params_bycol_Erb22daGMST.csv has been read and has length: 1230
/Users/jlanders/PycharmProjects/hol_temp_tsi_ccm_pb/hol_temp_tsi_ccm/Erb22daGMST_Wu18TSI/parameters/params_bycol_Erb22daGMST.csv duplicates have been dropped and now has length: 1230
Summary CSV file /Users/jlanders/PycharmProjects/hol_temp_tsi_ccm_pb/hol_temp_tsi_ccm/Erb22daGMST_Wu18TSI/parameters/summary_params_bycol_Erb22daGMST.csv has been created.


# Specified E-tau combinations

This second block targets a handful of specific (E, τ) pairs directly instead of a
dense grid — handy for spot-checking or rerunning individual configurations flagged
in earlier results. `lag_vals` spans the same ±40-decade scan used throughout (TSI
shifted by ℓ decades, −40 to +40, to find the alignment that maximizes CCM skill for
each configuration), and `surr_nums=[0, 101]` asks for 100 surrogate replicates per
direction for this batch (Text S1.3 uses 200 total per relationship across the full
pipeline).


In [7]:

# parameter_path = parameter_dir / f'{parameter_flag}.csv'
# parameter_df_master = pd.read_csv(parameter_path)

#[4, 1], [7,5],[9,2], 8,7]
# E_vals = [4]#np.arange(4, 11, 1)#[9,10]#[5]#[5,6, 7, 8, 9, 10,11] #3, 8
# tau_vals = [1]#, 5, 8]#[1,2]#,4,5,6]#[3]#[3, 4, 5, 6, 7]#, 6]#[6, 3, 8]

# E_vals = np.arange(4, 11, 1)
# tau_vals = np.arange(1, 9, 1)
tp_vals = [1]
knn_vals = [20]
lag_vals = np.arange(-40, 41, 2) #np.arange(0, 6, 1)
model = 'GISP2multiproxy'#['MPI', 'IPSL', 'CCSM3']#, 'MPI']#,'CCSM3',
col_var_ids = [model]#['MPI', 'IPSL', 'CCSM3']#, 'MPI']#,'CCSM3',
target_var_ids = ['vieira', 'wu']
surr_nums = [0, 101] # min and max surrogate numbers
param_csv_paths = []
for col_var_id in col_var_ids:
    for target_var_id in target_var_ids:
        for pair in [[4, 1], [7,5],[9,2], [8,7]]:
            E, tau = pair
            print(f'Processing E: {E}, tau: {tau}')
            for tp in tp_vals:
                for knn in knn_vals:
                    var_id_tuple, param_csv_path = general_lag_combinations(
                        col_var_id=col_var_id,
                        target_var_id=target_var_id,
                        E_vals=[E],
                        tau_vals=[tau],
                        lag_vals=lag_vals,
                        config=config,
                        parameters_d_master=parameters_d_master,
                        surr_nums=surr_nums,
                        surr_vars=config.surr_vars
                    )
                    print(var_id_tuple)
                    process_params_group(var_id_tuple)
                    param_csv_paths.append(param_csv_path)


Processing E: 4, tau: 1
('GISP2multiproxy', 'temp', 'vieira', 'TSI', ['temp', 'TSI'], [0, 101], [], PosixPath('/Users/jlanders/PycharmProjects/hol_temp_tsi_ccm/GISP2_straight/parameters/params_bycol_GISP2multiproxy.csv'), {'tau': {'values': [1]}, 'E': {'values': [4]}, 'train_len': {'values': [None]}, 'train_ind_i': {'values': [0]}, 'knn': {'values': [20]}, 'Tp_flag': {'values': [None]}, 'Tp': {'values': [1]}, 'lag': {'values': array([-40, -38, -36, -34, -32, -30, -28, -26, -24, -22, -20, -18, -16,
       -14, -12, -10,  -8,  -6,  -4,  -2,   0,   2,   4,   6,   8,  10,
        12,  14,  16,  18,  20,  22,  24,  26,  28,  30,  32,  34,  36,
        38,  40])}, 'Tp_lag_total': {'values': [32]}, 'sample': {'values': [100]}, 'weighted': {'values': [False]}, 'target_var': {'values': []}, 'col_var': {'values': []}, 'surr_var': {'values': ['neither']}, 'surr_num': {'values': [0]}})
Processing E: 7, tau: 5
('GISP2multiproxy', 'temp', 'vieira', 'TSI', ['temp', 'TSI'], [0, 101], [], PosixPath('/U

In [8]:
tidy_up_params(param_csv_paths, keep='first')


/Users/jlanders/PycharmProjects/hol_temp_tsi_ccm/GISP2_straight/parameters/params_bycol_GISP2multiproxy.csv has been read and has length: 1243998
/Users/jlanders/PycharmProjects/hol_temp_tsi_ccm/GISP2_straight/parameters/params_bycol_GISP2multiproxy.csv duplicates have been dropped and now has length: 1229522
Summary CSV file /Users/jlanders/PycharmProjects/hol_temp_tsi_ccm/GISP2_straight/parameters/summary_params_bycol_GISP2multiproxy.csv has been created.
